# Module 05 · Unit 03
# Zero-Trust Segmentation

| | |
|---|---|
| **Estimated time** | 65–75 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Attack Graphs & Reachability \[AG\] |
| **Refresher** | Module 05 · Unit 02 — Shortest Path and Attack Graphs |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. State and prove the **Edge Removal Theorem**: removing a critical path edge strictly increases the minimum attack cost
2. Define **zero-trust segmentation** formally in graph-theoretic terms
3. Distinguish between **edge hardening** (increasing weight) and **edge removal** (setting weight to $\infty$)
4. Implement a **greedy segmentation strategy** that maximises attack cost increase per edge removed
5. Compute the **minimum edge cut** — the fewest edges whose removal disconnects all paths from entry to target
6. Visualise the before-and-after attack graph and quantify the security improvement formally


## 🔣 Symbol Primer

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $\lambda(G, s, t)$ | Min edge cut | "min cut from $s$ to $t$" | Fewest edges whose removal disconnects $s$ from $t$ |
| $\kappa(G, s, t)$ | Min vertex cut | "min vertex cut" | Fewest vertices whose removal disconnects $s$ from $t$ |
| $G - e$ | Edge deletion | "$G$ minus edge $e$" | The graph $G$ with edge $e$ removed |
| $G - S$ | Edge set deletion | "$G$ minus $S$" | The graph $G$ with all edges in set $S$ removed |
| $\delta_w^{(G-S)}(s,t)$ | Post-cut cost | "shortest path cost after removing $S$" | Minimum cost in the reduced graph |
| $\Delta_S$ | Cost increase | "cost delta from removing $S$" | $\delta_w^{(G-S)}(s,t) - \delta_w^G(s,t)$ |

> **The module payoff.** Unit 01 computed what is reachable. Unit 02 computed the
> cheapest path to reach it. This unit answers the defender's question: *what is the
> minimum number of changes I need to make to this network to block the attack?*
> The answer is graph theory — specifically, the min-cut theorem.

---


## 1 · The Edge Removal Theorem

Before developing strategy, we need the fundamental theorem that justifies the
whole approach.

**Theorem (Edge Removal).** Let $G = (V, E, w)$ be a weighted directed graph,
$s, t \in V$, and let $P^*$ be a minimum-cost path from $s$ to $t$.
For any edge $e = (u, v) \in P^*$:

$$\delta_w^{(G-e)}(s, t) > \delta_w^G(s, t)$$

*Removing a critical path edge strictly increases the minimum attack cost.*

**Proof.**

Since $e \in P^*$, every minimum-cost path from $s$ to $t$ uses $e$ (otherwise
$P^*$ would not be a minimum-cost path — a cheaper alternative exists not using $e$).

After removing $e$, any path from $s$ to $t$ in $G - e$ must avoid $e$. The
minimum cost over all such paths is:

$$\delta_w^{(G-e)}(s,t) = \min_{P: s \rightsquigarrow t,\; e \notin P} \text{cost}(P)$$

Since $e \notin P$ for all $P$ in $G-e$, and the minimum-cost path in $G$ uses $e$,
the minimum over this restricted set is strictly greater than $\delta_w^G(s,t)$.

If no such path exists in $G-e$, then $\delta_w^{(G-e)}(s,t) = \infty > \delta_w^G(s,t)$.

In both cases: $\delta_w^{(G-e)}(s,t) > \delta_w^G(s,t)$. $\square$

### Corollary: Contrapositive Security Guarantee

By proof by contrapositive (Module 00):

$$\delta_w^{(G-e)}(s,t) = \delta_w^G(s,t) \;\Rightarrow\; e \notin P^*$$

*If removing an edge does not change the minimum cost, it was not on the critical path.*

This is the formal justification for the patch impact analysis from Unit 02:
an edge with zero cost delta contributes nothing to attacker difficulty and is
lower priority than one with positive delta.


## 2 · Zero-Trust Segmentation — Formal Definition

**Zero-trust architecture** is the security principle that no connection between
systems should be implicitly trusted — every access must be explicitly authorised
and verified. In graph-theoretic terms, it means systematically removing or
raising the cost of edges in the attack graph.

### Formal Definition

Given an attack graph $G = (V, E, w)$ with entry points $S \subseteq V$ and
targets $T \subseteq V$, a **zero-trust segmentation** is a set of edges
$C \subseteq E$ to remove such that:

$$\forall s \in S,\; \forall t \in T,\; t \notin R_{G-C}(s)$$

*After removing $C$, no target is reachable from any entry point.*

The **minimum segmentation** is the smallest such $C$ — the fewest edge removals
needed to achieve complete isolation. This is the **minimum edge cut** from
$S$ to $T$.

### Edge Hardening vs. Edge Removal

| Strategy | Graph operation | Effect |
|---|---|---|
| **Patch vulnerability** | Increase $w(u,v)$ | Raises attack cost, doesn't block |
| **Remove trust relationship** | Remove edge $(u,v)$ | $w(u,v) = \infty$; blocks that route |
| **Add authentication** | Increase $w(u,v)$ significantly | Raises cost; still traversable |
| **Network segmentation** | Remove edge $(u,v)$ | Breaks connectivity completely |

True zero-trust removes edges (or makes them prohibitively expensive — the
threshold where the rational attacker gives up). Partial hardening (increasing
weights) raises cost but does not guarantee isolation.

### The Min-Cut / Max-Flow Theorem

**Theorem (Max-Flow Min-Cut, Menger's Theorem).** The minimum number of edges
whose removal disconnects $s$ from $t$ equals the maximum number of
**edge-disjoint paths** from $s$ to $t$ — paths that share no edge.

This gives a lower bound on segmentation complexity: if there are $k$
edge-disjoint paths from $s$ to $t$, you must remove at least $k$ edges to
achieve isolation. Each edge-disjoint path is a distinct attack route that
must be individually blocked.


## 3 · Segmentation Strategy

With a limited budget (only $k$ edges can be removed or hardened), which $k$ edges
give the maximum increase in minimum attack cost?

### Greedy Segmentation Algorithm

A greedy approach: at each step, remove the edge that produces the largest increase
in minimum attack cost, then recompute.

```
GreedySegment(G, s, t, k):
  for i = 1 to k:
    best_edge = argmax_{e ∈ E} [δ_w(G-e, s, t) - δ_w(G, s, t)]
    remove best_edge from G
    if δ_w(G, s, t) = ∞: break   ← attack fully blocked
  return removed edges
```

**When greedy is optimal.** For a single target and single entry point, the greedy
algorithm is optimal for the first step (by the Edge Removal Theorem). For
subsequent steps, greedy may not find the globally optimal set — the problem of
finding the minimum-cost $k$-edge cut is NP-hard in general. However, greedy gives
a practical and interpretable approximation that works well for real networks.

### The Minimum Edge Cut

For complete isolation, the minimum edge cut $\lambda(G, s, t)$ gives the
theoretical minimum number of edge removals needed. NetworkX computes this via
max-flow (the Max-Flow Min-Cut theorem).

The minimum cut also identifies *which* edges to remove: the cut edges are the
bottleneck connecting $s$ to $t$ — the smallest set of connections whose loss
disconnects all attack paths.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[AG\]

| Concept | Security meaning |
|---|---|
| **Edge Removal Theorem** | Every patch on the critical path strictly improves security |
| **Zero-trust segmentation** | Systematic edge removal to isolate entry from target |
| **Min edge cut** | Minimum number of network controls needed for full isolation |
| **Edge-disjoint paths** | Distinct attack routes — each needs its own control |
| **Greedy segmentation** | Practical prioritised hardening roadmap |
| **$\delta_w = \infty$** | Complete isolation — the zero-trust ideal |
| **Cost threshold** | Attacker gives up when cost exceeds their budget |

**The formal statement of "defence-in-depth."** Defence-in-depth says that multiple
independent controls provide better security than one strong one. In graph terms:
multiple controls correspond to raising the cost of multiple edges on different
attack paths. The Edge Removal Theorem says each control independently contributes.
The Min-Cut theorem says the minimum number of independent controls needed equals
the number of edge-disjoint attack paths.

**The security guarantee this unit delivers.** After applying the minimum edge cut:

$$\forall s \in \text{EntryPoints},\; \forall t \in \text{Targets},\;
  \delta_w^{(G-C)}(s, t) = \infty$$

*No path from any entry to any target remains.* This is the strongest possible
isolation guarantee, expressed in precise mathematical terms. It was promised in
Module 00 (invariant), stated in Module 02 (formal security property), computed in
Unit 01 (reachability), costed in Unit 02 (Dijkstra), and achieved here.

---


## Python: Visualization & Verification

**1 · Edge Removal Theorem Verification** — for each edge on the minimum-cost path,
verify computationally that removing it strictly increases the attack cost, confirming
the theorem holds on our specific attack graph.

**2 · Greedy Segmentation** — implement the greedy strategy, show the cost increase
at each step, and visualise the network before and after segmentation.

**3 · Minimum Edge Cut** — compute the theoretical minimum number of removals for
complete isolation using max-flow; identify the cut edges; verify that removing
them makes $\delta_w = \infty$.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import heapq

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

def cvss_to_weight(cvss):
    return round(10.0 - cvss + 0.1, 2)

def dijkstra(G, source, weight_attr='weight'):
    INF  = float('inf')
    dist = {v: INF for v in G.nodes()}
    par  = {v: None for v in G.nodes()}
    dist[source] = 0.0
    pq = [(0.0, source)]
    done = set()
    while pq:
        d_u, u = heapq.heappop(pq)
        if u in done: continue
        done.add(u)
        for v in G.successors(u):
            w = G[u][v].get(weight_attr, 1)
            if d_u + w < dist[v]:
                dist[v] = d_u + w
                par[v]  = u
                heapq.heappush(pq, (dist[v], v))
    return dist, par

def reconstruct_path(parent, source, target):
    path, v = [], target
    while v is not None:
        path.append(v); v = parent[v]
    path.reverse()
    return path if path and path[0] == source else []

print('Setup complete.')


In [ ]:
# ── Rebuild the attack graph from Unit 02 ────────────────────────────────────

AG = nx.DiGraph()
nodes = {
    'internet':'external','web_server':'dmz','api_server':'dmz',
    'load_balancer':'dmz','db_primary':'internal','db_replica':'internal',
    'admin_workstation':'internal','dev_laptop':'internal',
    'log_server':'internal','backup_server':'internal',
    'domain_controller':'critical',
}
AG.add_nodes_from(nodes.keys())

cvss_edges = [
    ('internet','web_server',9.8), ('internet','api_server',8.1),
    ('web_server','load_balancer',5.3), ('web_server','api_server',7.5),
    ('api_server','db_primary',8.8), ('api_server','log_server',4.3),
    ('api_server','dev_laptop',6.5), ('load_balancer','db_primary',7.2),
    ('db_primary','db_replica',9.1), ('db_primary','backup_server',6.8),
    ('db_primary','admin_workstation',5.9), ('dev_laptop','admin_workstation',7.8),
    ('dev_laptop','log_server',4.1), ('admin_workstation','domain_controller',9.0),
    ('admin_workstation','log_server',3.5), ('log_server','backup_server',4.6),
]
for u, v, cvss in cvss_edges:
    AG.add_edge(u, v, cvss=cvss, weight=cvss_to_weight(cvss))

src, tgt = 'internet', 'domain_controller'
dist_base, par_base = dijkstra(AG, src)
base_cost = dist_base[tgt]
base_path = reconstruct_path(par_base, src, tgt)
path_edges = set(zip(base_path, base_path[1:]))

print(f"Baseline: {src} → {tgt}")
print(f"  Cost: {base_cost:.2f}")
print(f"  Path: {' → '.join(base_path)}")


### 1 · Edge Removal Theorem — Computational Verification

For each edge on the minimum-cost path, we remove it, re-run Dijkstra, and verify
that the cost strictly increased. We also check two off-path edges to confirm
they have no effect on the minimum cost.


In [ ]:
# ── 1 · Edge Removal Theorem Verification ────────────────────────────────────

print("Edge Removal Theorem Verification")
print("=" * 60)
print(f"Critical path: {' → '.join(base_path)}")
print(f"Baseline cost: {base_cost:.2f}")
print()

results = []
for u, v, data in AG.edges(data=True):
    G_test = AG.copy()
    G_test.remove_edge(u, v)
    d, _ = dijkstra(G_test, src)
    new_cost = d.get(tgt, float('inf'))
    on_path  = (u, v) in path_edges
    delta    = new_cost - base_cost
    results.append((u, v, data['cvss'], on_path, new_cost, delta))

results.sort(key=lambda x: (-x[4], x[0]))

print(f"  {'Edge':<38} {'On path':>8} {'New cost':>9} {'Δ cost':>8}  Theorem")
print("  " + "─" * 72)
for u, v, cvss, on_p, nc, delta in results:
    nc_str = f"{nc:.2f}" if nc < float('inf') else "∞"
    delta_str = f"+{delta:.2f}" if nc < float('inf') else "→ ∞"
    theorem = ("✓ δ INCREASED" if on_p and delta > 0
               else "✓ δ UNCHANGED (off path)" if not on_p and delta == 0
               else "✓ BLOCKED" if on_p and nc == float('inf')
               else "?")
    flag = "★" if on_p else " "
    print(f"  {flag} {u}→{v:<30} {'YES' if on_p else 'no':>8} "
          f"{nc_str:>9} {delta_str:>8}  {theorem}")

on_path_verified  = all(delta > 0 for u,v,_,on_p,nc,delta in results if on_p)
off_path_verified = all(delta == 0 for u,v,_,on_p,nc,delta in results if not on_p)
print()
print(f"All on-path edges: cost increased?  {on_path_verified} ✓")
print(f"All off-path edges: cost unchanged? {off_path_verified} ✓")
print(f"\nEdge Removal Theorem holds computationally on this attack graph. ✓")


**What do you see?**

Every edge on the critical path (marked ★) produces a positive cost delta or
completely blocks the attack ($\delta_w = \infty$). Every off-path edge produces
zero delta — exactly as the Edge Removal Theorem predicts.

The theorem is not just theoretically correct — it holds computationally on this
specific attack graph with real CVSS-derived weights. This is the pattern of
a formal proof verified by computation: the theorem provides the guarantee, the
code confirms it on the specific instance.

Notice which edge completely blocks the attack (cost → ∞). That is the
single most valuable security control in this network — it is the **bridge edge**:
the only edge on all paths between the entry point and the target. Removing it
is equivalent to installing a complete network segment boundary.


### 2 · Greedy Segmentation

We implement the greedy segmentation algorithm: at each step, remove the edge
producing the largest cost increase. We track the cost after each removal and
stop either when the budget is exhausted or the attack is completely blocked.


In [ ]:
# ── 2 · Greedy Segmentation Strategy ────────────────────────────────────────

def greedy_segment(G_orig, src, tgt, budget):
    """
    Greedy edge removal: iteratively remove the edge with highest cost delta.
    Returns list of (step, edge, old_cost, new_cost, delta).
    """
    G = G_orig.copy()
    history = []
    dist, par = dijkstra(G, src)
    current_cost = dist.get(tgt, float('inf'))

    for step in range(1, budget + 1):
        if current_cost == float('inf'):
            print(f"  Step {step}: Attack already blocked. Done.")
            break

        best_edge, best_delta, best_new = None, -1, current_cost
        for u, v in list(G.edges()):
            G_test = G.copy()
            G_test.remove_edge(u, v)
            d, _ = dijkstra(G_test, src)
            new_c = d.get(tgt, float('inf'))
            delta = (new_c - current_cost) if new_c < float('inf') else float('inf')
            if delta > best_delta:
                best_delta = delta
                best_edge  = (u, v)
                best_new   = new_c

        if best_edge is None:
            break

        G.remove_edge(*best_edge)
        old_cost     = current_cost
        current_cost = best_new
        history.append((step, best_edge, old_cost, current_cost,
                        best_delta if best_new < float('inf') else float('inf')))
        print(f"  Step {step}: Remove {best_edge[0]}→{best_edge[1]}  "
              f"cost: {old_cost:.2f} → "
              f"{'∞ (BLOCKED)' if current_cost == float('inf') else f'{current_cost:.2f}'}")

    return G, history

print(f"Greedy Segmentation: {src} → {tgt}")
print(f"Baseline cost: {base_cost:.2f}")
print()
AG_segmented, seg_history = greedy_segment(AG, src, tgt, budget=5)

# ── Visualise cost curve ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: cost progression
ax = axes[0]
steps  = [0] + [h[0] for h in seg_history]
costs  = [base_cost] + [h[3] if h[3] < float('inf') else None
                        for h in seg_history]
finite = [(s,c) for s,c in zip(steps,costs) if c is not None]
inf_at = [s for s,c in zip(steps,costs) if c is None]

if finite:
    ax.plot([s for s,c in finite], [c for s,c in finite],
            color=TS_BLUE, lw=2.5, marker='o', markersize=9, zorder=3)
    ax.fill_between([s for s,c in finite], [c for s,c in finite],
                    alpha=0.15, color=TS_BLUE)
if inf_at:
    ax.axvline(inf_at[0], color=TS_GREEN, lw=2, linestyle='--',
               label=f'Attack blocked at step {inf_at[0]}')
    ax.text(inf_at[0]+0.1, base_cost*0.6, '∞
(BLOCKED)',
            color=TS_GREEN, fontsize=11, fontweight='bold')

for s, c in finite:
    if s > 0:
        removed = seg_history[s-1][1]
        ax.text(s, c+0.3, f"{removed[0][:4]}→{removed[1][:4]}",
                ha='center', fontsize=7, color=TS_GRAY)

ax.set_xlabel('Segmentation step (edges removed)')
ax.set_ylabel('Minimum attack cost')
ax.set_title(f'Attack Cost vs Segmentation Steps
{src} → {tgt}',
             fontweight='bold', pad=8)
if inf_at:
    ax.legend(fontsize=9)

# Right: before/after network
ax2 = axes[1]
removed_edges = {h[1] for h in seg_history}
pos_g = nx.spring_layout(AG, seed=42, k=2.5)

nc = [TS_RED   if nodes[v] == 'critical' else
      TS_GREEN if v == src else TS_BLUE
      for v in AG.nodes()]

surviving = [(u,v) for u,v in AG.edges() if (u,v) not in removed_edges]
removed   = list(removed_edges)

nx.draw_networkx_edges(AG, pos_g, ax=ax2, edgelist=surviving,
                       edge_color=TS_GRAY, alpha=0.4, arrows=True,
                       arrowsize=14, width=1.5,
                       connectionstyle='arc3,rad=0.06')
if removed:
    nx.draw_networkx_edges(AG, pos_g, ax=ax2, edgelist=removed,
                           edge_color=TS_RED, alpha=0.7, arrows=True,
                           arrowsize=16, width=2.5, style='dashed',
                           connectionstyle='arc3,rad=0.15')
nx.draw_networkx_nodes(AG, pos_g, ax=ax2, node_color=nc,
                       node_size=800, alpha=0.92)
nx.draw_networkx_labels(AG, pos_g, ax=ax2, font_size=7,
                        font_color='white', font_weight='bold')

legend2 = [mpatches.Patch(color=TS_RED,  label='Removed edge (segmentation control)'),
           mpatches.Patch(color=TS_GRAY, label='Surviving edge')]
ax2.legend(handles=legend2, loc='upper left', fontsize=8)
ax2.set_title('Network After Greedy Segmentation
(dashed red = removed connections)',
              fontweight='bold', pad=8)
ax2.axis('off'); ax2.set_facecolor('white')

fig.patch.set_facecolor('white')
plt.suptitle('Zero-Trust Segmentation — Greedy Strategy',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**What do you see?**

The cost curve drops with each segmentation step and either reaches a plateau
(the attacker finds alternate paths) or jumps to infinity (all paths blocked).
The labels on the curve show which edge was removed at each step.

The network diagram highlights the removed edges in dashed red — these are the
network controls that were installed. Notice that the removed edges are precisely
the ones on the minimum-cost path that the Edge Removal Theorem identified as
critical. The greedy algorithm and the theorem agree.

**The operational translation.** Each removed edge corresponds to a specific
network control:
- Removing `internet → web_server` might mean: move the web server behind a WAF,
  require client certificates, or deploy an intrusion prevention system.
- Removing `admin_workstation → domain_controller` might mean: enforce Privileged
  Access Workstations (PAWs), require jump server access, disable direct domain
  admin connections.

The mathematics tells you *what* to do. The operations team implements it. The
graph tells you *why* it works.


### 3 · Minimum Edge Cut — The Theoretical Minimum

The greedy strategy gives a practical roadmap but may not find the true minimum
number of edges needed for complete isolation. NetworkX computes the minimum edge
cut via max-flow, giving us the theoretical lower bound and the optimal cut set.


In [ ]:
# ── 3 · Minimum Edge Cut ─────────────────────────────────────────────────────

# NetworkX min edge cut (unweighted — how many edges to remove to disconnect)
min_cut_val = nx.minimum_edge_cut(AG, src, tgt)
print(f"Minimum edge cut: {src} → {tgt}")
print(f"  Cut size: {len(min_cut_val)} edge(s)")
print(f"  Cut edges: {sorted(min_cut_val)}")
print()
print(f"  Interpretation: {len(min_cut_val)} independent attack route(s) exist.")
print(f"  You must block ALL {len(min_cut_val)} to achieve complete isolation.")

# Verify: removing cut edges disconnects the attack
AG_cut = AG.copy()
for u, v in min_cut_val:
    AG_cut.remove_edge(u, v)

d_cut, _ = dijkstra(AG_cut, src)
cost_after_cut = d_cut.get(tgt, float('inf'))
print(f"\nCost after removing minimum cut: {cost_after_cut}")
print(f"Target unreachable: {cost_after_cut == float('inf')} ✓")

# Compare with Menger's theorem: edge-disjoint paths
edge_disjoint = list(nx.edge_disjoint_paths(AG, src, tgt))
print(f"\nMenger's theorem check:")
print(f"  Edge-disjoint paths from {src} to {tgt}: {len(edge_disjoint)}")
print(f"  Min edge cut:                             {len(min_cut_val)}")
print(f"  Equal (Max-Flow Min-Cut holds):           {len(edge_disjoint) == len(min_cut_val)} ✓")
print(f"\nEdge-disjoint attack paths (each must be independently blocked):")
for i, path in enumerate(edge_disjoint, 1):
    print(f"  Path {i}: {' → '.join(path)}")

# ── Final visualisation: before/after with min cut ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
pos_final = nx.spring_layout(AG, seed=42, k=2.8)

def node_col(v):
    if v == src:   return TS_GREEN
    if v == tgt:   return TS_RED
    return TS_BLUE

nc_final = [node_col(v) for v in AG.nodes()]

for ax, graph, title, cut_set in [
    (axes[0], AG,     f'Before Segmentation
Cost({src}→{tgt}) = {base_cost:.2f}', set()),
    (axes[1], AG_cut, f'After Min-Cut Segmentation
Cost({src}→{tgt}) = ∞ (BLOCKED)', min_cut_val),
]:
    cut_edges   = [(u,v) for u,v in AG.edges() if (u,v) in cut_set]
    other_edges = [(u,v) for u,v in AG.edges() if (u,v) not in cut_set
                   and graph.has_edge(u,v)]
    crit_edges  = [(u,v) for u,v in AG.edges() if (u,v) in path_edges
                   and (u,v) not in cut_set and graph.has_edge(u,v)]

    nx.draw_networkx_edges(AG, pos_final, ax=ax, edgelist=other_edges,
                           edge_color=TS_GRAY, alpha=0.35, arrows=True,
                           arrowsize=14, width=1.5,
                           connectionstyle='arc3,rad=0.06')
    if crit_edges:
        nx.draw_networkx_edges(AG, pos_final, ax=ax, edgelist=crit_edges,
                               edge_color=TS_AMBER, alpha=0.8, arrows=True,
                               arrowsize=18, width=2.5,
                               connectionstyle='arc3,rad=0.06')
    if cut_edges:
        for u,v in cut_edges:
            nx.draw_networkx_edges(AG, pos_final, ax=ax, edgelist=[(u,v)],
                                   edge_color=TS_RED, alpha=0.9, arrows=True,
                                   arrowsize=20, width=3.5, style='dashed',
                                   connectionstyle='arc3,rad=0.25')

    nx.draw_networkx_nodes(AG, pos_final, ax=ax, node_color=nc_final,
                           node_size=900, alpha=0.92)
    nx.draw_networkx_labels(AG, pos_final, ax=ax, font_size=7,
                            font_color='white', font_weight='bold')
    ax.set_title(title, fontweight='bold', pad=10, fontsize=10)
    ax.axis('off'); ax.set_facecolor('white')

legend_final = [
    mpatches.Patch(color=TS_GREEN, label='Entry point'),
    mpatches.Patch(color=TS_RED,   label='Target / cut edge'),
    mpatches.Patch(color=TS_AMBER, label='Critical path edge'),
    mpatches.Patch(color=TS_GRAY,  label='Other edge'),
]
fig.legend(handles=legend_final, loc='lower center', ncol=4,
           fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.patch.set_facecolor('white')
plt.suptitle('Zero-Trust Segmentation: Before and After Minimum Edge Cut',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**What do you see?**

The minimum edge cut tells you the theoretical minimum number of network controls
needed to completely isolate the entry point from the target. The Max-Flow Min-Cut
theorem is verified computationally: the number of edge-disjoint paths equals the
minimum cut size.

The before/after visualisation makes the security transformation concrete:
- **Before**: the attack graph has multiple paths to the target, the amber edges
  highlighting the cheapest route.
- **After**: the dashed red cut edges are removed; Dijkstra confirms the cost is
  now $\infty$ — no path exists.

**The formal security guarantee is now TRUE:**

$$\forall s \in \text{EntryPoints},\; \forall t \in \text{Targets},\;
  \delta_w^{(G-C)}(s, t) = \infty$$

The network has been formally verified as isolated with respect to the defined
entry points and targets. This is the mathematical foundation of a zero-trust
architecture review — not a checklist, not a best-practice document, but a
proof that the specific network topology achieves the specific security property.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. MINIMUM VERTEX CUT
#    nx.minimum_node_cut(G, src, tgt) finds the minimum number of VERTICES
#    (not edges) whose removal disconnects src from tgt.
#    Compare the minimum vertex cut to the minimum edge cut for our attack graph.
#    Which is smaller? What does removing a vertex mean operationally
#    (vs removing an edge)?
#
# 2. MULTI-ENTRY MULTI-TARGET ISOLATION
#    We've been solving the single-entry single-target problem.
#    For multiple entry points E = {internet, dev_laptop} and
#    multiple targets T = {domain_controller, db_primary}:
#    Find the minimum set of edges whose removal ensures NO entry point can
#    reach ANY target. (Hint: add a super-source connected to all entries and
#    a super-target connected from all targets, then find the min cut on the
#    augmented graph.)
#
# 3. EDGE HARDENING VS REMOVAL
#    Instead of removing edges (cost → ∞), simulate hardening them:
#    increase CVSS-derived weight by a fixed factor (e.g., 2x) rather than
#    removing. Compare:
#      - How many edges must be hardened (2x) to reach cost > 15?
#      - How many edges must be removed to reach cost = ∞?
#    Which strategy is more practical? Which provides a stronger guarantee?
#    Plot both strategies on the same cost-vs-controls chart.

# Your work here:


---

## Summary

| Concept | Definition | Security meaning |
|---|---|---|
| **Edge Removal Theorem** | Removing a critical path edge strictly raises cost | Every on-path patch directly improves security |
| **Zero-trust segmentation** | Remove edges until all paths blocked | Systematic network isolation |
| **Edge hardening** | Increase $w(e)$ | Raises cost; doesn't guarantee isolation |
| **Edge removal** | $w(e) \to \infty$ | Breaks connectivity; guarantees isolation |
| **Min edge cut $\lambda$** | Fewest edges to disconnect $s$ from $t$ | Minimum network controls for full isolation |
| **Max-Flow Min-Cut** | Min cut = max edge-disjoint paths | Each independent attack route needs its own control |
| **Greedy segmentation** | Remove highest-delta edge iteratively | Practical prioritised hardening roadmap |

---

## Module 05 Complete

| Unit | Content | Payoff |
|---|---|---|
| **01** | BFS, DFS, reachability, formal audit | What is reachable; formal isolation guarantee |
| **02** | Dijkstra, CVSS weights, patch ranking | Minimum-cost attack path; patch prioritisation |
| **03** | Edge removal theorem, min cut, segmentation | How to block the attack; formal proof it works |

The three units form one operational argument: **compute what is reachable, find
the cheapest path, remove the minimum controls to block it, prove it is blocked.**
This is the complete mathematical foundation of network attack graph analysis —
from the first edge in the graph to the formal isolation guarantee.

---

## Up Next

**Module 06 — Trees and Recursion**

Trees are the graphs with no cycles and exactly $|V|-1$ edges. They appear
everywhere: decision trees in classifiers, parse trees in compilers and prompt
analysis, recursive neural architectures. Module 06 connects tree structure to
the security decisions made at every node.

$\rightarrow$ `modules/module-06/unit-01-trees-definitions.ipynb`
